In [19]:
import pandas as pd
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt
from datetime import datetime

In [27]:
NEEDED = [
    'gameId','date','isHome','teamName', 'teamID','oppTeamName', 'oppTeamID',
    'fieldGoalsAttempted','offensiveRebounds','turnovers','freeThrowsAttempted',
    'threePointsMade','fieldGoalsMade','freeThrowsMade'
]

df = pd.read_csv('data/cbb_raw.csv', usecols = NEEDED)
df['possessions'] = (df['fieldGoalsAttempted'] - df['offensiveRebounds']) + df['turnovers'] + (0.475 * df['freeThrowsAttempted'])
df['pointsScored'] = (3 * df['threePointsMade']) + (2 * (df['fieldGoalsMade'] - df['threePointsMade']) + (df['freeThrowsMade']))

a = df['teamName']
b = df['oppTeamName']
df['matchup'] = np.where(a < b, a + ' vs ' + b, b + ' vs ' + a)

df['avgPossessions'] = df.groupby(by=['matchup', 'date'])['possessions'].transform('mean')
df['pointsAllowed'] = df.groupby(by=['matchup', 'date'])['pointsScored'].transform('sum') - df['pointsScored']

df['margin'] = df['pointsScored'] - df['pointsAllowed']
df['total_points'] = df['pointsScored'] + df['pointsAllowed']
df['marginCapped'] = df['margin'].clip(-16,16)
df['pointsScoredCapped'] = (df['total_points'] + df['marginCapped']) / 2
df['pointsAllowedCapped'] = (df['total_points'] - df['marginCapped']) / 2

df['offensiveEfficiency'] = df['pointsScoredCapped'] * 100 / df['possessions']
df['defensiveEfficiency'] = df['pointsAllowedCapped'] * 100 / df['possessions']

df2 = df.copy()
df2['side'] = np.where(df2['isHome'], 'home', 'away')

# 2) pivot: one row per (gameId, date), columns split by side
wide = (
    df2
    .set_index(['gameId', 'date', 'side'])
    .drop(columns=['isHome', 'matchup'], errors='ignore') 
    .unstack('side')
)

# 3) flatten MultiIndex columns like ('teamID','home') -> 'homeTeamID'
wide.columns = [f"{side}{col[0].upper()}{col[1:]}" for col, side in wide.columns]
wide = wide.reset_index()

wide['homeOffensiveEfficiency'] = wide['homeOffensiveEfficiency'] - 3.75
wide['homeDefensiveEfficiency'] = wide['homeDefensiveEfficiency'] + 3.75
wide['awayOffensiveEfficiency'] = wide['awayOffensiveEfficiency'] + 3.75
wide['awayDefensiveEfficiency'] = wide['awayDefensiveEfficiency'] - 3.75

nationalAverageEfficiency = 100 * (wide['homePointsScored'].sum() + wide['awayPointsScored'].sum()) / (wide['homeAvgPossessions'].sum() * 2)

wide.sort_values(by='date', ascending=True, inplace=True)
wide

,gameId,date,awayTeamID,homeTeamID,awayTeamName,homeTeamName,awayOppTeamID,homeOppTeamID,awayOppTeamName,homeOppTeamName,...,awayMarginCapped,homeMarginCapped,awayPointsScoredCapped,homePointsScoredCapped,awayPointsAllowedCapped,homePointsAllowedCapped,awayOffensiveEfficiency,homeOffensiveEfficiency,awayDefensiveEfficiency,homeDefensiveEfficiency
0,6500992,2025-11-03,2654,489,Southern Miss.,Buffalo,489,2654,Buffalo,Southern Miss.,...,-6,6,79.0,85.0,85.0,79.0,122.546992,126.818356,124.069549,125.101767
4016,6505053,2025-11-03,1560,53,Bellarmine,Georgia,53,1560,Georgia,Bellarmine,...,-16,16,73.5,89.5,89.5,73.5,100.651780,112.521517,114.246045,99.235547
4067,6505105,2025-11-03,1388165,1918,Paine,Kennesaw St.,1918,1388165,Kennesaw St.,Paine,...,-16,16,59.0,75.0,75.0,59.0,78.575618,93.024194,91.367311,79.879032
1082,6502092,2025-11-03,720,1876,Coastal Carolina,Western Mich.,1876,720,Western Mich.,Coastal Carolina,...,-5,5,71.0,76.0,76.0,71.0,99.052013,95.434339,98.263423,96.409054
4076,6505114,2025-11-03,1169,2159,Missouri,Howard,2159,1169,Howard,Missouri,...,16,-16,85.5,69.5,69.5,85.5,124.214952,91.980028,94.171803,121.518595
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6181,6593988,2026-03-15,494,611,Wichita St.,South Fla.,611,494,South Fla.,Wichita St.,...,-15,15,55.0,70.0,70.0,55.0,85.231481,97.809666,99.953704,83.546881
6136,6593745,2026-03-15,2680,113,Vanderbilt,Arkansas,113,2680,Arkansas,Vanderbilt,...,-11,11,75.0,86.0,86.0,75.0,110.701872,118.888146,118.888146,110.701872
6155,6593943,2026-03-15,1294,508,Penn,Yale,508,1294,Yale,Penn,...,4,-4,88.0,84.0,84.0,88.0,123.315217,114.435016,110.380435,127.562874
5898,6537881,2026-03-15,2118,1362,Dayton,VCU,1362,2118,VCU,Dayton,...,-8,8,62.0,70.0,70.0,62.0,100.511607,103.900903,105.496976,99.097943


In [58]:
def recency_weighted_avg(grades, half_life=12):
    g = np.asarray(grades, dtype=float)
    n = len(g)
    if n == 0:
        return np.nan

    # newest gets exponent 0, previous gets 1, ..., oldest gets n-1
    k = np.arange(n-1, -1, -1)
    weights = 0.5 ** (k / half_life)
    return np.sum(g * weights) / np.sum(weights)

def safe_current(team, series_dict, cache_dict, half_life, default = nationalAverageEfficiency):
    if team in cache_dict:
        return cache_dict[team]
    if len(series_dict[team]) == 0:
        return default
    cache_dict[team] = recency_weighted_avg(series_dict[team], half_life)
    return cache_dict[team]

adjustedOffensiveEfficiencies = defaultdict(list)
adjustedDefensiveEfficiencies = defaultdict(list)

curOff = {}  # team -> weighted avg of adjOffEffs[team]
curDef = {}  # team -> weighted avg of adjDefEffs[team]

rows_by_game = []

for r in wide.itertuples(index=False):
    home = r.homeTeamName
    away = r.awayTeamName

    # opponent efficiencies
    away_def = safe_current(away, adjustedDefensiveEfficiencies, curDef, 12, default=nationalAverageEfficiency)
    home_def = safe_current(home, adjustedDefensiveEfficiencies, curDef, 12, default=nationalAverageEfficiency)
    away_off = safe_current(away, adjustedOffensiveEfficiencies, curOff, 12, default=nationalAverageEfficiency)
    home_off = safe_current(home, adjustedOffensiveEfficiencies, curOff, 12, default=nationalAverageEfficiency)

    # home court advantage adjustments
    home_adj_off = (r.homeOffensiveEfficiency * nationalAverageEfficiency) / away_def
    away_adj_off = (r.awayOffensiveEfficiency * nationalAverageEfficiency) / home_def
    home_adj_def = (r.homeDefensiveEfficiency * nationalAverageEfficiency) / away_off
    away_adj_def = (r.awayDefensiveEfficiency * nationalAverageEfficiency) / home_off

    if len(adjustedOffensiveEfficiencies[home]) == 0: home_adj_off = r.homeOffensiveEfficiency
    if len(adjustedOffensiveEfficiencies[away]) == 0: away_adj_off = r.awayOffensiveEfficiency
    if len(adjustedDefensiveEfficiencies[home]) == 0: home_adj_def = r.homeDefensiveEfficiency
    if len(adjustedDefensiveEfficiencies[away]) == 0: away_adj_def = r.awayDefensiveEfficiency

    # append
    adjustedOffensiveEfficiencies[home].append(home_adj_off)
    adjustedOffensiveEfficiencies[away].append(away_adj_off)
    adjustedDefensiveEfficiencies[home].append(home_adj_def)
    adjustedDefensiveEfficiencies[away].append(away_adj_def)

    curOff.pop(home, None); curOff.pop(away, None)
    curDef.pop(home, None); curDef.pop(away, None)

    home_post_off = recency_weighted_avg(adjustedOffensiveEfficiencies[home])
    home_post_def = recency_weighted_avg(adjustedDefensiveEfficiencies[home])
    away_post_off = recency_weighted_avg(adjustedOffensiveEfficiencies[away])
    away_post_def = recency_weighted_avg(adjustedDefensiveEfficiencies[away])

    rows_by_game.append({
        "gameId": r.gameId, "date": r.date, "team": home, "opponent": away, "isHome": True,
        "adjOff_pre": home_off, "adjDef_pre": home_def, "adjNet_pre": home_off - home_def,
        "adjOff_post": home_post_off, "adjDef_post": home_post_def, "adjNet_post": home_post_off - home_post_def,
    })
    rows_by_game.append({
        "gameId": r.gameId, "date": r.date, "team": away, "opponent": home, "isHome": False,
        "adjOff_pre": away_off, "adjDef_pre": away_def, "adjNet_pre": away_off - away_def,
        "adjOff_post": away_post_off, "adjDef_post": away_post_def, "adjNet_post": away_post_off - away_post_def,
    })

# whole-season summaries
wholeSeasonAdjOffEffs = [
    (recency_weighted_avg(vals), team, len(vals))
    for team, vals in adjustedOffensiveEfficiencies.items()
]

wholeSeasonAdjDefEffs = [
    (recency_weighted_avg(vals), team, len(vals))
    for team, vals in adjustedDefensiveEfficiencies.items()
]

In [5]:
df_by_game = pd.DataFrame(rows_by_game)
df_by_game

,gameId,date,team,opponent,isHome,adjOff_pre,adjDef_pre,adjNet_pre,adjOff_post,adjDef_post,adjNet_post
0,6500992,2025-11-03,Buffalo,Southern Miss.,True,108.020933,108.020933,0.000000,126.818356,125.101767,1.716590
1,6500992,2025-11-03,Southern Miss.,Buffalo,False,108.020933,108.020933,0.000000,122.546992,124.069549,-1.522556
2,6505053,2025-11-03,Georgia,Bellarmine,True,108.020933,108.020933,0.000000,112.521517,99.235547,13.285969
3,6505053,2025-11-03,Bellarmine,Georgia,False,108.020933,108.020933,0.000000,100.651780,114.246045,-13.594265
4,6505105,2025-11-03,Kennesaw St.,Paine,True,108.020933,108.020933,0.000000,93.024194,79.879032,13.145161
...,...,...,...,...,...,...,...,...,...,...,...
12385,6593943,2026-03-15,Penn,Yale,False,109.444111,100.272951,9.171159,110.661359,100.191958,10.469401
12386,6537881,2026-03-15,VCU,Dayton,True,120.575591,102.248136,18.327455,120.276490,101.845963,18.430527
12387,6537881,2026-03-15,Dayton,VCU,False,111.404733,96.758927,14.645807,111.064048,96.612257,14.451791
12388,6593954,2026-03-15,Michigan,Purdue,True,121.292268,94.179330,27.112938,120.901366,95.246530,25.654836


In [ ]:
df_new_year = df_by_game[df_by_game['date'] >= '2026-01-01']

In [ ]:
def find_adjNet_pre(df_new_year, team_name, date):
    return float(df_new_year[(df_new_year['team'] == team_name) & (df_new_year['date'] == date)]['adjNet_pre'].iloc[0])

In [ ]:
def evaluate_kenpom_model(raw_fn, margin_cap, home_court_advantage, half_life):
    df = pd.read_csv(raw_fn, usecols = NEEDED)
    df['possessions'] = (df['fieldGoalsAttempted'] - df['offensiveRebounds']) + df['turnovers'] + (0.475 * df['freeThrowsAttempted'])
    df['pointsScored'] = (3 * df['threePointsMade']) + (2 * (df['fieldGoalsMade'] - df['threePointsMade']) + (df['freeThrowsMade']))

    a = df['teamName']
    b = df['oppTeamName']
    df['matchup'] = np.where(a < b, a + ' vs ' + b, b + ' vs ' + a)

    df['avgPossessions'] = df.groupby(by=['matchup', 'date'])['possessions'].transform('mean')
    df['pointsAllowed'] = df.groupby(by=['matchup', 'date'])['pointsScored'].transform('sum') - df['pointsScored']

    df['margin'] = df['pointsScored'] - df['pointsAllowed']
    df['total_points'] = df['pointsScored'] + df['pointsAllowed']
    df['marginCapped'] = df['margin'].clip(-margin_cap, margin_cap)
    df['pointsScoredCapped'] = (df['total_points'] + df['marginCapped']) / 2
    df['pointsAllowedCapped'] = (df['total_points'] - df['marginCapped']) / 2

    df['offensiveEfficiency'] = df['pointsScoredCapped'] * 100 / df['possessions']
    df['defensiveEfficiency'] = df['pointsAllowedCapped'] * 100 / df['possessions']

    df2 = df.copy()
    df2['side'] = np.where(df2['isHome'], 'home', 'away')

    # 2) pivot: one row per (gameId, date), columns split by side
    wide = (
        df2
        .set_index(['gameId', 'date', 'side'])
        .drop(columns=['isHome', 'matchup'], errors='ignore')  # optional drops
        .unstack('side')
    )

    # 3) flatten MultiIndex columns like ('teamID','home') -> 'homeTeamID'
    wide.columns = [f"{side}{col[0].upper()}{col[1:]}" for col, side in wide.columns]
    wide = wide.reset_index()

    wide['homeOffensiveEfficiency'] = wide['homeOffensiveEfficiency'] - home_court_advantage
    wide['homeDefensiveEfficiency'] = wide['homeDefensiveEfficiency'] + home_court_advantage
    wide['awayOffensiveEfficiency'] = wide['awayOffensiveEfficiency'] + home_court_advantage
    wide['awayDefensiveEfficiency'] = wide['awayDefensiveEfficiency'] - home_court_advantage

    nationalAverageEfficiency = 100 * (wide['homePointsScored'].sum() + wide['awayPointsScored'].sum()) / (wide['homeAvgPossessions'].sum() * 2)

    wide.sort_values(by='date', ascending=True, inplace=True)

    adjustedOffensiveEfficiencies = defaultdict(list)
    adjustedDefensiveEfficiencies = defaultdict(list)

    curOff = {}  # team -> weighted avg of adjOffEffs[team]
    curDef = {}  # team -> weighted avg of adjDefEffs[team]

    rows_by_game = []

    for r in wide.itertuples(index=False):
        home = r.homeTeamName
        away = r.awayTeamName

        # opponent "strength" estimates (fallback to nationalAverage if no history)
        away_def = safe_current(away, adjustedDefensiveEfficiencies, curDef, half_life, default=nationalAverageEfficiency)
        home_def = safe_current(home, adjustedDefensiveEfficiencies, curDef, half_life, default=nationalAverageEfficiency)
        away_off = safe_current(away, adjustedOffensiveEfficiencies, curOff, half_life, default=nationalAverageEfficiency)
        home_off = safe_current(home, adjustedOffensiveEfficiencies, curOff, half_life, default=nationalAverageEfficiency)

        # your adjustments
        home_adj_off = (r.homeOffensiveEfficiency * nationalAverageEfficiency) / away_def
        away_adj_off = (r.awayOffensiveEfficiency * nationalAverageEfficiency) / home_def
        home_adj_def = (r.homeDefensiveEfficiency * nationalAverageEfficiency) / away_off
        away_adj_def = (r.awayDefensiveEfficiency * nationalAverageEfficiency) / home_off

        # if you want the "first game is raw" behavior, keep it:
        if len(adjustedOffensiveEfficiencies[home]) == 0: home_adj_off = r.homeOffensiveEfficiency
        if len(adjustedOffensiveEfficiencies[away]) == 0: away_adj_off = r.awayOffensiveEfficiency
        if len(adjustedDefensiveEfficiencies[home]) == 0: home_adj_def = r.homeDefensiveEfficiency
        if len(adjustedDefensiveEfficiencies[away]) == 0: away_adj_def = r.awayDefensiveEfficiency

        # append
        adjustedOffensiveEfficiencies[home].append(home_adj_off)
        adjustedOffensiveEfficiencies[away].append(away_adj_off)
        adjustedDefensiveEfficiencies[home].append(home_adj_def)
        adjustedDefensiveEfficiencies[away].append(away_adj_def)

        # update caches ONLY for teams that changed (O(1) cache invalidation)
        curOff.pop(home, None); curOff.pop(away, None)
        curDef.pop(home, None); curDef.pop(away, None)

        home_post_off = recency_weighted_avg(adjustedOffensiveEfficiencies[home], half_life)
        home_post_def = recency_weighted_avg(adjustedDefensiveEfficiencies[home], half_life)
        away_post_off = recency_weighted_avg(adjustedOffensiveEfficiencies[away], half_life)
        away_post_def = recency_weighted_avg(adjustedDefensiveEfficiencies[away], half_life)

        rows_by_game.append({
            "gameId": r.gameId, "date": r.date, "team": home, "opponent": away, "isHome": True,
            "adjOff_pre": home_off, "adjDef_pre": home_def, "adjNet_pre": home_off - home_def,
            "adjOff_post": home_post_off, "adjDef_post": home_post_def, "adjNet_post": home_post_off - home_post_def,
        })
        rows_by_game.append({
            "gameId": r.gameId, "date": r.date, "team": away, "opponent": home, "isHome": False,
            "adjOff_pre": away_off, "adjDef_pre": away_def, "adjNet_pre": away_off - away_def,
            "adjOff_post": away_post_off, "adjDef_post": away_post_def, "adjNet_post": away_post_off - away_post_def,
        })

    df_by_game = pd.DataFrame(rows_by_game)
    df_new_year = df_by_game[df_by_game['date'] >= '2026-01-01']    

    wide_new_year = wide.loc[wide['date'] >= '2026-01-01'][['gameId', 'date', 'awayTeamName', 'homeTeamName', 'homeMargin']].copy()
    wide_new_year['home_adjNet_pre'] = wide_new_year.apply(lambda r: find_adjNet_pre(df_new_year, r.homeTeamName, r.date), axis=1)
    wide_new_year['away_adfNet_pre'] = wide_new_year.apply(lambda r: find_adjNet_pre(df_new_year, r.awayTeamName, r.date), axis=1)
    wide_new_year['predicted_homeMargin'] = wide_new_year['home_adjNet_pre'] - wide_new_year['away_adfNet_pre'] + home_court_advantage
    wide_new_year['error'] = abs(wide_new_year['predicted_homeMargin'] - wide_new_year['homeMargin'])   

    return wide_new_year['error'].mean()

In [ ]:
hyper_param_rows = []
for mc in range(8,18,2):
    for hca in np.arange(1, 3.5, 0.5):
        for hl in range(16, 32, 2):
            avg_error = evaluate_kenpom_model(raw_fn='data/cbb_raw.csv', margin_cap=mc, home_court_advantage=hca, half_life=hl)
            hyper_param_rows.append(
                {
                    'margin_cap': mc,
                    'home_court_advantage': hca,
                    'half_life': hl,
                    'avg_error': avg_error
                }
            )

hyper_param_df = pd.DataFrame(hyper_param_rows)

In [76]:
hyper_param_df[hyper_param_df['avg_error'] == hyper_param_df.avg_error.min()].drop_duplicates()

,margin_cap,home_court_advantage,half_life,avg_error
179,12,2.0,20,9.068014


In [77]:
def get_rankings(raw_fn, margin_cap, home_court_advantage, half_life):
    df = pd.read_csv(raw_fn, usecols = NEEDED)
    df['possessions'] = (df['fieldGoalsAttempted'] - df['offensiveRebounds']) + df['turnovers'] + (0.475 * df['freeThrowsAttempted'])
    df['pointsScored'] = (3 * df['threePointsMade']) + (2 * (df['fieldGoalsMade'] - df['threePointsMade']) + (df['freeThrowsMade']))

    a = df['teamName']
    b = df['oppTeamName']
    df['matchup'] = np.where(a < b, a + ' vs ' + b, b + ' vs ' + a)

    df['avgPossessions'] = df.groupby(by=['matchup', 'date'])['possessions'].transform('mean')
    df['pointsAllowed'] = df.groupby(by=['matchup', 'date'])['pointsScored'].transform('sum') - df['pointsScored']

    df['margin'] = df['pointsScored'] - df['pointsAllowed']
    df['total_points'] = df['pointsScored'] + df['pointsAllowed']
    df['marginCapped'] = df['margin'].clip(-margin_cap, margin_cap)
    df['pointsScoredCapped'] = (df['total_points'] + df['marginCapped']) / 2
    df['pointsAllowedCapped'] = (df['total_points'] - df['marginCapped']) / 2

    df['offensiveEfficiency'] = df['pointsScoredCapped'] * 100 / df['possessions']
    df['defensiveEfficiency'] = df['pointsAllowedCapped'] * 100 / df['possessions']

    df2 = df.copy()
    df2['side'] = np.where(df2['isHome'], 'home', 'away')

    # 2) pivot: one row per (gameId, date), columns split by side
    wide = (
        df2
        .set_index(['gameId', 'date', 'side'])
        .drop(columns=['isHome', 'matchup'], errors='ignore')  # optional drops
        .unstack('side')
    )

    # 3) flatten MultiIndex columns like ('teamID','home') -> 'homeTeamID'
    wide.columns = [f"{side}{col[0].upper()}{col[1:]}" for col, side in wide.columns]
    wide = wide.reset_index()

    wide['homeOffensiveEfficiency'] = wide['homeOffensiveEfficiency'] - home_court_advantage
    wide['homeDefensiveEfficiency'] = wide['homeDefensiveEfficiency'] + home_court_advantage
    wide['awayOffensiveEfficiency'] = wide['awayOffensiveEfficiency'] + home_court_advantage
    wide['awayDefensiveEfficiency'] = wide['awayDefensiveEfficiency'] - home_court_advantage

    nationalAverageEfficiency = 100 * (wide['homePointsScored'].sum() + wide['awayPointsScored'].sum()) / (wide['homeAvgPossessions'].sum() * 2)

    wide.sort_values(by='date', ascending=True, inplace=True)

    adjustedOffensiveEfficiencies = defaultdict(list)
    adjustedDefensiveEfficiencies = defaultdict(list)

    curOff = {}  # team -> weighted avg of adjOffEffs[team]
    curDef = {}  # team -> weighted avg of adjDefEffs[team]

    rows_by_game = []

    for r in wide.itertuples(index=False):
        home = r.homeTeamName
        away = r.awayTeamName

        # opponent "strength" estimates (fallback to nationalAverage if no history)
        away_def = safe_current(away, adjustedDefensiveEfficiencies, curDef, half_life, default=nationalAverageEfficiency)
        home_def = safe_current(home, adjustedDefensiveEfficiencies, curDef, half_life, default=nationalAverageEfficiency)
        away_off = safe_current(away, adjustedOffensiveEfficiencies, curOff, half_life, default=nationalAverageEfficiency)
        home_off = safe_current(home, adjustedOffensiveEfficiencies, curOff, half_life, default=nationalAverageEfficiency)

        # your adjustments
        home_adj_off = (r.homeOffensiveEfficiency * nationalAverageEfficiency) / away_def
        away_adj_off = (r.awayOffensiveEfficiency * nationalAverageEfficiency) / home_def
        home_adj_def = (r.homeDefensiveEfficiency * nationalAverageEfficiency) / away_off
        away_adj_def = (r.awayDefensiveEfficiency * nationalAverageEfficiency) / home_off

        # if you want the "first game is raw" behavior, keep it:
        if len(adjustedOffensiveEfficiencies[home]) == 0: home_adj_off = r.homeOffensiveEfficiency
        if len(adjustedOffensiveEfficiencies[away]) == 0: away_adj_off = r.awayOffensiveEfficiency
        if len(adjustedDefensiveEfficiencies[home]) == 0: home_adj_def = r.homeDefensiveEfficiency
        if len(adjustedDefensiveEfficiencies[away]) == 0: away_adj_def = r.awayDefensiveEfficiency

        # append
        adjustedOffensiveEfficiencies[home].append(home_adj_off)
        adjustedOffensiveEfficiencies[away].append(away_adj_off)
        adjustedDefensiveEfficiencies[home].append(home_adj_def)
        adjustedDefensiveEfficiencies[away].append(away_adj_def)

        # update caches ONLY for teams that changed (O(1) cache invalidation)
        curOff.pop(home, None); curOff.pop(away, None)
        curDef.pop(home, None); curDef.pop(away, None)

        home_post_off = recency_weighted_avg(adjustedOffensiveEfficiencies[home], half_life)
        home_post_def = recency_weighted_avg(adjustedDefensiveEfficiencies[home], half_life)
        away_post_off = recency_weighted_avg(adjustedOffensiveEfficiencies[away], half_life)
        away_post_def = recency_weighted_avg(adjustedDefensiveEfficiencies[away], half_life)

        rows_by_game.append({
            "gameId": r.gameId, "date": r.date, "team": home, "opponent": away, "isHome": True,
            "adjOff_pre": home_off, "adjDef_pre": home_def, "adjNet_pre": home_off - home_def,
            "adjOff_post": home_post_off, "adjDef_post": home_post_def, "adjNet_post": home_post_off - home_post_def,
        })
        rows_by_game.append({
            "gameId": r.gameId, "date": r.date, "team": away, "opponent": home, "isHome": False,
            "adjOff_pre": away_off, "adjDef_pre": away_def, "adjNet_pre": away_off - away_def,
            "adjOff_post": away_post_off, "adjDef_post": away_post_def, "adjNet_post": away_post_off - away_post_def,
        })

    # whole-season summaries
    wholeSeasonAdjOffEffs = [
        (recency_weighted_avg(vals, half_life), team, len(vals))
        for team, vals in adjustedOffensiveEfficiencies.items()
    ]

    wholeSeasonAdjDefEffs = [
        (recency_weighted_avg(vals, half_life), team, len(vals))
        for team, vals in adjustedDefensiveEfficiencies.items()
    ]

    off_df = pd.DataFrame(wholeSeasonAdjOffEffs, columns=['adjustedOffensiveEfficiencies', 'team', 'games'])
    def_df = pd.DataFrame(wholeSeasonAdjDefEffs, columns=['adjustedDefensiveEfficiencies', 'team', 'games'])

    team_df = (
        off_df.merge(def_df, on='team', how='outer', suffixes=('_off', '_def'))
            .rename(columns={'games_off': 'games_off', 'games_def': 'games_def'})
    )

    # optional: if you expect games counts to match, keep one
    team_df['games'] = team_df[['games_off', 'games_def']].max(axis=1)
    team_df = team_df.drop(columns=['games_off', 'games_def'])

    team_df = team_df.sort_values('adjustedOffensiveEfficiencies', ascending=False).reset_index(drop=True)
    team_df = team_df[team_df['games'] >= 10]

    team_df['adjustedNetRating'] = team_df['adjustedOffensiveEfficiencies'] - team_df['adjustedDefensiveEfficiencies']

    team_df = team_df.sort_values(by='adjustedNetRating', ascending=False).reset_index(drop=True)

    return team_df

best_rankings = get_rankings(raw_fn='data/cbb_raw.csv', margin_cap=12, home_court_advantage=2, half_life=20)


In [81]:
pd.set_option('display.max_rows', 100)
best_rankings.head(100)

,adjustedOffensiveEfficiencies,team,adjustedDefensiveEfficiencies,games,adjustedNetRating
0,121.363033,Duke,97.530670,34,23.832363
1,120.497460,Arizona,96.842419,34,23.655041
2,119.540596,Michigan,96.719817,34,22.820778
3,126.252821,Purdue,104.973198,35,21.279623
4,118.226476,Houston,97.423385,34,20.803091
5,115.390347,St. John's (NY),95.480524,34,19.909824
6,123.658343,Illinois,103.990080,32,19.668263
7,116.741845,Gonzaga,97.321708,33,19.420137
8,120.161558,Saint Mary's (CA),100.777616,32,19.383943
9,117.293790,UConn,97.912478,34,19.381312
